### ML Practical 1: Uber Ride Price Prediction (Aligned with Faculty Notebook)
**Tasks:**
1. Pre-process the dataset & engineer features (`distance_km`, `Year`, `Month`, `Day`, `Hour`)
2. Identify and remove outliers
3. Check correlation
4. Implement Linear Regression and Random Forest Regression models
5. Evaluate and compare models using MAE, RMSE, and R2

In [ ]:
# Cell 1: Install haversine (run once if not installed)
!pip install haversine

In [ ]:
# Cell 2: Import Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from haversine import haversine, Unit
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [ ]:
# Cell 3: Load & Inspect Dataset
df = pd.read_csv("uber.csv")
df.head()

In [ ]:
# Cell 4: Drop Unused Columns & Missing Values
df.drop(['Unnamed: 0', 'key'], axis=1, inplace=True)
df.dropna(inplace=True)
df.isnull().sum()

In [ ]:
# Cell 5: Filter Valid Coordinates & Positive Fare Amounts
df = df[
    (df["pickup_latitude"] >= -90) & (df["pickup_latitude"] <= 90) &
    (df["dropoff_latitude"] >= -90) & (df["dropoff_latitude"] <= 90) &
    (df["pickup_longitude"] >= -180) & (df["pickup_longitude"] <= 180) &
    (df["dropoff_longitude"] >= -180) & (df["dropoff_longitude"] <= 180)
]
df = df[df["fare_amount"] > 0]
df.shape

In [ ]:
# Cell 6: Feature Engineering - Haversine Distance (km)
df["distance_km"] = df.apply(
    lambda row: haversine(
        (row["pickup_latitude"], row["pickup_longitude"]),
        (row["dropoff_latitude"], row["dropoff_longitude"]),
        unit=Unit.KILOMETERS
    ),
    axis=1
)
# Keep realistic Uber ride distances (0 < distance <= 100 km)
df = df[(df["distance_km"] > 0) & (df["distance_km"] <= 100)]
df.head()

In [ ]:
# Cell 7: Identify Outliers (Scatter Plot & Boxplot Before Removal)
plt.scatter(df['distance_km'], df['fare_amount'])
plt.xlabel('Distance in km')
plt.ylabel('Fare Amount')
plt.title('Fare Amount vs Distance (Before Outlier Removal)')
plt.show()

sns.boxplot(y=df['fare_amount'])
plt.title('Fare Amount Boxplot (Before Outlier Removal)')
plt.show()

In [ ]:
# Cell 8: Remove Fare Outliers using IQR
Q1 = df["fare_amount"].quantile(0.25)
Q3 = df["fare_amount"].quantile(0.75)
IQR = Q3 - Q1
Lower = Q1 - 1.5 * IQR
Upper = Q3 + 1.5 * IQR

df = df[(df["fare_amount"] >= Lower) & (df["fare_amount"] <= Upper)]
print("Dataset Shape After Removing Outliers:", df.shape)

sns.boxplot(y=df['fare_amount'])
plt.title('Fare Amount Boxplot (After Outlier Removal)')
plt.show()

In [ ]:
# Cell 9: Extract Datetime Features (Year, Month, Day, Hour)
df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'])
df['Year'] = df['pickup_datetime'].dt.year
df['Month'] = df['pickup_datetime'].dt.month
df['Day'] = df['pickup_datetime'].dt.day
df['Hour'] = df['pickup_datetime'].dt.hour
df.drop('pickup_datetime', axis=1, inplace=True)
df.head()

In [ ]:
# Cell 10: Check Correlation Matrix
corr = df.corr(numeric_only=True)
plt.figure(figsize=(12, 8))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Matrix")
plt.show()

In [ ]:
# Cell 11: Train-Test Split
X = df.drop("fare_amount", axis=1)
y = df["fare_amount"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

In [ ]:
# Cell 12: Linear Regression Model & Evaluation
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_prediction = lr.predict(X_test)

mae_lr = mean_absolute_error(y_test, lr_prediction)
rmse_lr = np.sqrt(mean_squared_error(y_test, lr_prediction))
r2_lr = r2_score(y_test, lr_prediction)

print("Linear Regression MAE :", mae_lr)
print("Linear Regression RMSE:", rmse_lr)
print("Linear Regression R2  :", r2_lr)

In [ ]:
# Cell 13: Random Forest Regression Model & Evaluation
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_prediction = rf.predict(X_test)

mae_rf = mean_absolute_error(y_test, rf_prediction)
rmse_rf = np.sqrt(mean_squared_error(y_test, rf_prediction))
r2_rf = r2_score(y_test, rf_prediction)

print("Random Forest MAE :", mae_rf)
print("Random Forest RMSE:", rmse_rf)
print("Random Forest R2  :", r2_rf)